# CellChat differential signaling (Figure 4D)

CellChat identified infection-induced rewiring of signaling networks, marked by a pronounced increase in interactions between myeloid and CD8⁺ T cell populations (Fig. 4D). Circle plot depicts differential signaling between infected and uninfected conditions, with red edges indicating interactions increased during infection and blue edges indicating interactions decreased during infection. Edge thickness corresponds to the aggregated strength of ligand–receptor interactions between sender and receiver cell type pairs.

Inputs (from `cellchat_fig4_table1.Rmd`): `cellchat_edges_uninfected.csv`, `cellchat_edges_infected.csv`, `cellchat_proper_metadata.csv`.



## 1. Setup


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch, Circle
from matplotlib.patheffects import withStroke

IN_DIR = Path(os.environ.get("CELLCHAT_OUT", "outputs/cellchat"))
OUT_DIR = Path(os.environ.get("FIG4D_OUT", "outputs/cellchat"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

EDGES_UNINF = IN_DIR / "cellchat_edges_uninfected.csv"
EDGES_INF = IN_DIR / "cellchat_edges_infected.csv"
META = IN_DIR / "cellchat_proper_metadata.csv"


## 2. Load edges and aggregate to coarse groups

Sum fine-type interaction weights within each coarse pair, then take infected − uninfected.


In [ ]:
edges_uninfected = pd.read_csv(EDGES_UNINF)
edges_infected = pd.read_csv(EDGES_INF)
meta = pd.read_csv(META)

fine_to_coarse = dict(zip(meta["cell_type"], meta["coarse_cluster"]))
coarse_to_fine = meta.groupby("coarse_cluster")["cell_type"].apply(list).to_dict()
coarse_groups = list(coarse_to_fine.keys())

cellchat_cells = set(edges_uninfected["from"]) | set(edges_uninfected["to"]) | set(
    edges_infected["from"]
) | set(edges_infected["to"])
missing = cellchat_cells - set(fine_to_coarse)
if missing:
    print(f"WARNING: {len(missing)} cell types missing from metadata:", sorted(missing)[:10])
else:
    print("All CellChat cell types mapped to coarse groups.")


def coarse_strength(edges_df, name):
    e = edges_df.copy()
    e["from_coarse"] = e["from"].map(fine_to_coarse)
    e["to_coarse"] = e["to"].map(fine_to_coarse)
    e = e.dropna(subset=["from_coarse", "to_coarse"])
    e = e[e["weight"] > 0]
    strength = e.groupby(["from_coarse", "to_coarse"], as_index=False)["weight"].sum()
    print(f"{name}: {len(e)} fine edges → {len(strength)} coarse pairs")
    return strength


def strength_matrix(strength_df, groups):
    mat = pd.DataFrame(0.0, index=groups, columns=groups)
    for _, row in strength_df.iterrows():
        mat.loc[row["from_coarse"], row["to_coarse"]] = row["weight"]
    return mat


s_un = coarse_strength(edges_uninfected, "Uninfected")
s_inf = coarse_strength(edges_infected, "Infected")
mat_un = strength_matrix(s_un, coarse_groups)
mat_inf = strength_matrix(s_inf, coarse_groups)
mat_diff = mat_inf - mat_un

rows = []
for a in coarse_groups:
    for b in coarse_groups:
        if a == b:
            continue
        diff = float(mat_diff.loc[a, b])
        rows.append(
            {
                "from": a,
                "to": b,
                "strength_uninfected": float(mat_un.loc[a, b]),
                "strength_infected": float(mat_inf.loc[a, b]),
                "difference": diff,
                "abs_difference": abs(diff),
            }
        )

differential_df = pd.DataFrame(rows)
differential_df.to_csv(OUT_DIR / "fig4d_coarse_differential_edges.csv", index=False)
print(differential_df.head())
print(
    "Net strength change:",
    f"{mat_diff.to_numpy().sum():+.4f}",
    "| edges:",
    len(differential_df),
)


## 3. Circle plot (Figure 4D)


In [ ]:
def create_coarse_circle_plot(differential_df, coarse_groups):
    fig, ax = plt.subplots(figsize=(14, 14), facecolor="white", dpi=300)
    n_groups = len(coarse_groups)
    theta = np.linspace(0, 2 * np.pi, n_groups, endpoint=False)
    radius = 0.7
    x = [radius * np.cos(a) for a in theta]
    y = [radius * np.sin(a) for a in theta]

    colors = plt.cm.tab20(np.linspace(0, 1, 20))[:n_groups]
    if n_groups > 20:
        colors = np.vstack(
            [colors, plt.cm.tab20b(np.linspace(0, 1, n_groups - 20))]
        )
    cell_colors = {cell: colors[i] for i, cell in enumerate(coarse_groups)}

    max_abs = differential_df["abs_difference"].max() if len(differential_df) else 1.0

    for _, row in differential_df.iterrows():
        i = coarse_groups.index(row["from"])
        j = coarse_groups.index(row["to"])
        difference = row["difference"]
        abs_diff = row["abs_difference"]

        if abs_diff < 1e-10:
            edge_width = 0.3
            edge_color = mcolors.to_rgba("#CCCCCC", alpha=0.3)
            mutation_scale = 8
            z = 1
        else:
            norm = abs_diff / max_abs if max_abs > 0 else 0
            edge_width = 0.8 + 3.5 * norm
            mutation_scale = 12 + 8 * norm
            if difference > 0:
                edge_color = mcolors.to_rgba("#D62728", alpha=0.4 + 0.5 * norm)
            else:
                edge_color = mcolors.to_rgba("#1F77B4", alpha=0.4 + 0.5 * norm)
            z = 1 + norm

        ax.add_patch(
            FancyArrowPatch(
                (x[i], y[i]),
                (x[j], y[j]),
                connectionstyle="arc3,rad=0.2",
                arrowstyle="-|>",
                mutation_scale=mutation_scale,
                lw=edge_width,
                color=edge_color,
                zorder=z,
            )
        )

    for i, cell in enumerate(coarse_groups):
        ax.add_patch(
            Circle(
                (x[i], y[i]),
                0.035,
                fill=True,
                color=cell_colors[cell],
                edgecolor="white",
                linewidth=1.5,
                zorder=10,
            )
        )

    label_radius = 0.85
    for i, cell in enumerate(coarse_groups):
        angle = theta[i]
        label_x = label_radius * np.cos(angle)
        label_y = label_radius * np.sin(angle)
        rotation_deg = np.degrees(angle)
        if 90 <= rotation_deg <= 270:
            rotation_deg -= 180
        ax.text(
            label_x,
            label_y,
            cell.replace("_", " "),
            fontsize=11,
            rotation=rotation_deg,
            rotation_mode="anchor",
            ha="center",
            va="center",
            color="black",
            zorder=11,
            family="Arial",
        )

    ax.text(
        0,
        1.35,
        "Communication Rewiring During Infection",
        fontsize=16,
        fontweight="bold",
        ha="center",
        va="center",
        color="#2C3E50",
        family="Arial",
        zorder=15,
        path_effects=[withStroke(linewidth=3, foreground="white")],
    )
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.6)
    ax.set_aspect("equal")
    ax.axis("off")
    return fig, ax


fig, ax = create_coarse_circle_plot(differential_df, coarse_groups)
fig.savefig(OUT_DIR / "fig4d_communication_rewiring.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(OUT_DIR / "fig4d_communication_rewiring.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved →", OUT_DIR / "fig4d_communication_rewiring.pdf")


## Notes

- Edge CSVs are generated in `cellchat_fig4_table1.Rmd` from `cellchat@net$weight`.  
- Point `CELLCHAT_OUT` at that export folder before running this notebook.
